# PROJET DE TRAITEMENTS DISTRIBUES
Edoardo PICIUCCHI, Aurélien DUVIGNAC-ROSA, Jean-Marc FAUVEL

# Présentation

Le projet consiste en l'étude d'un article intitulé : « CCF : Fast and Scalable Connected Component Computation in MapReduce » (« CCF : Calcul de composantes connexes rapide et scalable en MapReduce »). Cet article a été publié en 2013 par Jimmy Lin et Michael Schatz. Il propose un algorithme de calcul de composantes connexes en utilisant le modèle de programmation MapReduce. Cet algorithme est basé sur l'algorithme de Label Propagation. Il est conçu pour être rapide et scalable, c'est-à-dire qu'il peut être utilisé sur de grands ensembles de données et sur un grand nombre de machines.

L'algorithme a pour objectif d’identifier les sous-graphes connectés au sein d’un graphe G. Le graphe analysé doit être représenté par une collection de paires (N1, N2) où N1 et N2 sont des noeuds du graph G qui ont une connexion entre eux. Le couple (N1, N2) représente une arête du graphe.
L’algorithme présenté dans ce papier va permettre d’identifier des sous-graphes connectés entre eux en modifiant les paires des sous-graphes afin qu’elles soient toutes constituées ainsi : (N, Nmin) où N est un noeud du sous graph et Nmin le noeud de plus petite valeur appartenant au même sous-graphe.
Par ailleurs, l’algorithme ayant pour objet de permettre de traiter des graphes de grande taille, il adopte une programmation distribuée de type MapReduce, permettant ainsi de répartir le traitement sur plusieurs machines et permettre le traitement des données de manière parallèle.

## Objectifs du projet

1. Lire, comprendre et expliquer l’algorithme décrit dans le papier de Hakan Kardes, Siddharth Agrawal, Xin Wang et Ang Sun intitulé « CCF : Fast and Scalable Connected Component Computation in MapReduce » ;
2. Coder l’algorithme en Spark en utilisant à la fois des RDD et des DataFrames ;
3. L’implémentation doit être exécutée en Python, et optionnellement en Scala
4. Effectuer une analyse comparative des versions en RDD et DataFrame sur des graphes de tailles croissantes ;
5. Utilisation de DataBricks pour les petits graphes, et Google Cloud Cluster pour les plus gros (<20 Gb).

La première étape a consisté à implémenter l'algorithme CCF en se limitant exclusivement à l'utilisation de python sans le paradigme spark. Cela nous a permis de mieux appréhender l'algorithme et d'en comprendre les différentes étapes. Cette étape réalisée, nous avons pu utiliser les éléments relatifs à la bibiliothèque `spark`.
L'implémentation décrite ci-dessous permet de réaliser les étapes de l'algorithme CCF décrites dans l'exemple de l'article.
Les résultats obtenus nous ont permis de constater une erreur au niveau de l'exemple. En effet, dans la figure 5.1 de l'article, le lien entre H et G n'est pas transmis dans le graphe présent dans la colonne *reducer*. Pour le prouver il suffit d'ajouter le paramètre `debug` dans le constructeur de l'instance du graph utilisé de sorte à pouvoir afficher les différentes étapes de l'algorithme et notamment les graphes successifs obtenus après chaque itération. Remarquons à cet égard que les graphes successifs sont décrits par l'intermédiaire d'un dictionnaire.
Comme nous pouvons le constater à l'issue de la première itération il manque le lien entre H et G dans l'exemple de l'article.
Cependant le résultat final obtenu est bien identique à celui fourni dans l'article.

In [3]:
class Graph:
    """
    Graphe exemple
    """

    def __init__(self, input_dict=None):
        self.graph = input_dict or {}

    def __str__(self):
        graph_sorted = {key: self.graph[key] for key in sorted(self.graph)}
        s = ""
        for key, values in graph_sorted.items():
            for value in values:
                s += f"{key} -> {value} \n"
        return s

    def __eq__(self, other):
        return self.graph == other.graph

    def ccf(self, debug=False):
        """
        Algorithme CCF
        """
        # Initialiser le nombre d'itérations
        iterations = 1
        previous_Graph = (
            self  # Initialisation pour stocker le graphe précédent
        )

        while True:
            # Effectuer les étapes mapper et reducer
            current_Graph = self.mapper(previous_Graph)
            current_Graph = self.reducer(current_Graph, debug)
            if debug:
                print(f"itération {iterations} : \n{current_Graph}")

            # Comparer le graphe précédent et le graphe actuel
            if previous_Graph == current_Graph:
                # Si les graphes sont identiques, arrêter la boucle
                if debug:
                    print(
                        f"Convergence atteinte après {iterations} itérations."
                    )
                return current_Graph, iterations

            # Mettre à jour le graphe précédent
            previous_Graph = current_Graph
            iterations += 1

    def mapper(self, input_Graph):
        """
        Créer un nouveau dictionnaire pour stocker le graphe bidirectionnel
        """
        bidirectional_graph = {}
        # Parcourir les nœuds du graphe original
        for node, neighbors in input_Graph.graph.items():
            # Ajouter chaque voisin à la liste des voisins du nœud courant
            if node not in bidirectional_graph:
                bidirectional_graph[node] = []
            for neighbor in neighbors:
                if neighbor not in bidirectional_graph:
                    bidirectional_graph[neighbor] = []
                # Ajouter les voisins dans les deux sens
                if neighbor not in bidirectional_graph[node]:
                    bidirectional_graph[node].append(neighbor)
                if node not in bidirectional_graph[neighbor]:
                    bidirectional_graph[neighbor].append(node)
        return Graph(bidirectional_graph)

    def map(self, key, value):
        """
        Fonction map
        """
        return f"""	emit({key},{value})
								emit({value},{key})
						"""

    def reduce(self, key, values, debug=False):
        # Initialiser la liste des valeurs et le compteur
        valueList = []
        CounterNewPair = 0
        min = key

        # Dictionnaire pour stocker les émissions
        emissions = {}

        # Trouver la valeur minimale et remplir valueList
        for value in values:
            if value < min:
                min = value
            valueList.append(value)

        # Vérifier si une émission est nécessaire
        if min < key:
            # Ajouter l'émission (key, min) au dictionnaire
            if key not in emissions:
                emissions[key] = []
            emissions[key].append(min)

            if debug:
                print(f"emit({key},{min})")

            # Ajouter les émissions pour les autres valeurs dans valueList
            for value in valueList:
                if min != value:
                    CounterNewPair += 1
                    if debug:
                        print(f"emit({value},{min})")

                    if value not in emissions:
                        emissions[value] = []
                    emissions[value].append(min)

        # Retourner ou enregistrer les émissions pour usage ultérieur
        return emissions

    def reducer(self, input_Graph, debug=False):
        emissions = []
        for key, value in input_Graph.graph.items():
            emissions.append(self.reduce(key, value, debug))
        return Graph(self.merge_dictionaries(emissions))

    def merge_dictionaries(self, dictionaries):
        # Dictionnaire final pour stocker les résultats
        final_dict = {}

        # Parcourir les dictionnaires successifs
        for current_dict in dictionaries:
            for key, values in current_dict.items():
                # Si la clef n'existe pas, initialiser une liste vide
                if key not in final_dict:
                    final_dict[key] = []
                # Ajouter les valeurs, en évitant les doublons
                for value in values:
                    if value not in final_dict[key]:
                        final_dict[key].append(value)
        return final_dict


# Exemple de graphe
input_dict = {
    "A": ["B"],
    "B": ["C", "D"],
    "D": ["E"],
    "F": ["G"],
    "G": ["H"],
}

debug = True
g = Graph(input_dict)
ccf_Graph, iterations = g.ccf(debug=debug)
print(
    f"A partir du graphe initial :\n{g}\net à l'issue de {iterations} itérations, le graphe suivant a été obtenu :\n{ccf_Graph}"
)

emit(B,A)
emit(C,A)
emit(D,A)
emit(C,B)
emit(D,B)
emit(E,B)
emit(E,D)
emit(G,F)
emit(H,F)
emit(H,G)
itération 1 : 
B -> A 
C -> A 
C -> B 
D -> A 
D -> B 
E -> B 
E -> D 
G -> F 
H -> F 
H -> G 

emit(B,A)
emit(C,A)
emit(D,A)
emit(E,A)
emit(C,A)
emit(B,A)
emit(D,A)
emit(B,A)
emit(E,A)
emit(E,B)
emit(D,B)
emit(G,F)
emit(H,F)
emit(H,F)
emit(G,F)
itération 2 : 
B -> A 
C -> A 
D -> A 
D -> B 
E -> A 
E -> B 
G -> F 
H -> F 

emit(B,A)
emit(D,A)
emit(E,A)
emit(C,A)
emit(D,A)
emit(B,A)
emit(E,A)
emit(B,A)
emit(G,F)
emit(H,F)
itération 3 : 
B -> A 
C -> A 
D -> A 
E -> A 
G -> F 
H -> F 

emit(B,A)
emit(D,A)
emit(E,A)
emit(C,A)
emit(G,F)
emit(H,F)
itération 4 : 
B -> A 
C -> A 
D -> A 
E -> A 
G -> F 
H -> F 

Convergence atteinte après 4 itérations.
A partir du graphe initial :
A -> B 
B -> C 
B -> D 
D -> E 
F -> G 
G -> H 

et à l'issue de 4 itérations, le graphe suivant a été obtenu :
B -> A 
C -> A 
D -> A 
E -> A 
G -> F 
H -> F 



## Implémentation de l'algorithme CCF en paradigme Spark

### Importation des librairies

In [ ]:
from pyspark import SparkContext, SparkConf

### Création de la classe GraphSpart

La classe `GraphSpark` permet de représenter un graphe. Elle est composée de deux attributs :
- `graph_rdd` : un dictionnaire dont les clefs sont les noeuds du graphe et les valeurs sont les noeuds auxquels ils sont connectés ;
- `sc` : le contexte spark.

La classe `GraphSpark` est composée de plusieurs méthodes :
- `__init__` : le constructeur de la classe ;
- `__str__` : permet d'afficher le graphe ;
- `mapper` : permet de mapper les noeuds du graphe en créant un graphe bidirectionnel ;
- `reducer` : permet de réduire le graphe en fusionnant les noeuds connectés ;
- `ccf` : permet d'appliquer l'algorithme CCF sur le graphe ;
- `stop` : permet de stopper le contexte spark.

In [ ]:
class GraphSpark:
    def __init__(self, input_dict=None, sc=None):
        """
        Initialise le graphe avec un dictionnaire d'entrée ou un RDD.
        """
        self.sc = sc or SparkContext.getOrCreate(
            SparkConf().setAppName("GraphCCF").setMaster("local")
        )
        self.graph_rdd = (
            self.sc.parallelize(input_dict.items())
            if input_dict
            else self.sc.parallelize([])
        )

    def __str__(self):
        """
        Représentation textuelle du graphe.
        """
        graph = self.graph_rdd.collectAsMap()
        s = ""
        for key, values in sorted(graph.items()):
            for value in values:
                s += f"{key} -> {value}\n"
        return s

    def mapper(self):
        """
        Étape Mapper : Crée un graphe bidirectionnel.
        """

        def map_edges(node_neighbors):
            node, neighbors = node_neighbors
            edges = []
            for neighbor in neighbors:
                edges.append((node, neighbor))
                edges.append((neighbor, node))
            return edges

        self.graph_rdd = (
            self.graph_rdd.flatMap(map_edges).groupByKey().mapValues(list)
        )

    def reducer(self):
        """
        Étape Reducer : Applique la logique de réduction CCF.
        """

        def reduce_node(node_neighbors):
            key, neighbors = node_neighbors
            min_node = key
            value_list = list(neighbors)

            # Trouver la valeur minimale
            for neighbor in neighbors:
                if neighbor < min_node:
                    min_node = neighbor

            emissions = []
            if min_node < key:
                # Émettre la relation (key, min_node)
                emissions.append((key, min_node))
                # Émettre pour les autres valeurs
                for neighbor in value_list:
                    if neighbor != min_node:
                        emissions.append((neighbor, min_node))
            return emissions

        # Appliquer la réduction à chaque nœud
        self.graph_rdd = (
            self.graph_rdd.flatMap(reduce_node)
            .groupByKey()
            .mapValues(lambda x: list(set(x)))
        )

    def ccf(self, debug=False):
        """
        Algorithme principal CCF : boucle jusqu'à convergence.
        """
        iterations = 1
        previous_graph = None

        while True:
            if debug:
                print(f"\n--- Iteration {iterations} ---")
                print(f"{self.graph_rdd.collectAsMap()}")
                print(f"{self}")

            # Étape Mapper
            self.mapper()

            # Étape Reducer
            self.reducer()

            # Vérifier la convergence
            current_graph = self.graph_rdd.collectAsMap()
            if previous_graph == current_graph:
                if debug:
                    print(
                        f"Convergence atteinte après {iterations} itérations."
                    )
                break

            previous_graph = current_graph
            iterations += 1

    def stop(self):
        """
        Arrête SparkContext.
        """
        self.sc.stop()

### Exemple d'utilisation de la classe

Pour illustrer l'utilisation de la classe `GraphSpark`, nous allons utiliser l'exemple de l'article. Cet exemple est composé de 6 noeuds et 6 arêtes. Les noeuds sont les suivants : A, B, C, D, E et F. Les arêtes sont les suivantes : (A, B), (B, C), (C, D), (D, E), (E, F) et (F, A).

In [ ]:
# Exemple d'utilisation
if __name__ == "__main__":
    input_dict = {
        "A": ["B"],
        "B": ["C", "D"],
        "D": ["E"],
        "F": ["G"],
        "G": ["H"],
    }

    debug = True

    # Créer une instance de GraphSpark
    sc = SparkContext.getOrCreate(
        SparkConf().setAppName("GraphCCF").setMaster("local")
    )
    g = GraphSpark(input_dict, sc)

    # Exécuter l'algorithme CCF
    g.ccf(debug=debug)

    g.stop()

### Explication du code

`self.sc = sc or SparkContext.getOrCreate(SparkConf().setAppName("GraphCCF").setMaster("local"))` permet de créer un contexte spark si celui-ci n'existe pas déjà.

`self.graph_rdd = self.sc.parallelize(graph).flatMap(self.mapper).groupByKey().mapValues(list)` permet de créer un RDD à partir du graphe. Ce RDD est composé de paires (noeud, noeud connecté). Les noeuds connectés sont regroupés par noeud.

`self.graph_rdd = self.graph_rdd.flatMap(map_edges).groupByKey().mapValues(list)` : cette ligne permet de mapper les nœuds du graphe en créant un graphe bidirectionnel. En effet, pour chaque noeud du graphe, une paire (noeud, noeud connecté) et une paire (noeud connecté, noeud) sont créées. Les nœuds connectés sont ensuite regroupés en nœud.

- `flatMap(map_edges)` génère toutes les relations bidirectionnelles `(key, value)` ;
- `groupByKey()` regroupe les voisins d’un même nœud sous la même clef ;
- `mapValues(list)` transforme ces voisins en une liste pour assurer une manipulation plus aisée.

1. `flatMap(map_edges)`:
	- `flatMap` permet d'une part d'appliquer la fonction `map_edges` à chaque élément du RDD et d'autre part de renvoyer une liste de paires ;
	- `map_edges` prend en paramètre une paire (nœud, nœud connecté) et renvoie une liste de paires [(nœud, nœud connecté), (nœud connecté, nœud)]. Elle est utilisée pour créer des relations bidirectionnelles (par exemple, à partir du lien le A -> B, `map_edges` génère A -> B et B -> A).

Pour illustrer le propos, exploitons le graphe composé des relations suivantes : (A, B), (B, C), (B, D).
`map_edges` va générer les relations suivantes : (A, B), (B, A), (B, C), (C, B), (B, D), (D, B).

In [ ]:
# Avant flatMap
graph_rdd = [("A", ["B"]), ("B", ["C", "D"])]

# Après flatMap avec map_edges
graph_rdd = [
    ("A", "B"),
    ("B", "A"),
    ("B", "C"),
    ("C", "B"),
    ("B", "D"),
    ("D", "B"),
]

2. `groupByKey()`:
	- `groupByKey` permet de regrouper les valeurs associées à chaque clef ;
	- ici, les noeuds connectés sont regroupés par noeud.

In [ ]:
# Avant groupByKey
graph_rdd = [
    ("A", "B"),
    ("B", "A"),
    ("B", "C"),
    ("C", "B"),
    ("B", "D"),
    ("D", "B"),
]

# Après groupByKey
graph_rdd = [("A", ["B"]), ("B", ["A", "C", "D"]), ("C", ["B"]), ("D", ["B"])]

3. `mapValues(list)`:

	- `mapValues` permet d'appliquer la fonction `list` à chaque valeur du RDD ;
	- ici, la fonction `list` est appliquée pour transformer les valeurs regroupées en liste.

`self.graph_rdd = self.graph_rdd.flatMap(reduce_node).groupByKey().mapValues(lambda x: list(set(x)))` permet de réduire le graphe en fusionnant les noeuds connectés.

1. `flatMap(reduce_node)`:
	- `flatMap` permet d'une part d'appliquer la fonction `reduce_node` à chaque élément du RDD et d'autre part de renvoyer une liste de paires ;
	- `reduce_node` prend en paramètre une paire (nœud, nœud connecté) et renvoie une liste de paires [(nœud, nœud connecté), (nœud connecté, nœud)]. Elle est utilisée pour créer des relations bidirectionnelles (par exemple, à partir du lien le A -> B, `reduce_node` génère A -> B et B -> A).

2. `groupByKey()`:
	- `groupByKey` permet de regrouper les valeurs associées à chaque clef ;
	- ici, les noeuds connectés sont regroupés par noeud.

3. `mapValues(lambda x: list(set(x)))`:
	- Applique une fonction pour dédupliquer les valeurs associées à chaque clef ;
	- `set(x)` permet de supprimer les doublons ;
	- `list` permet de transformer le set en liste.

`current_graph = self.graph_rdd.collectAsMap()` permet de collecter les données du RDD (Resilient Distributed Dataset) sous la forme d'un dictionnaire Python.

1. `self.graph_rdd` est un RDD composé de paires (noeud, noeud connecté) qui représente les données du graphe.
2. `collectAsMap()` permet de collecter les données de l'ensemble des partitions du RDD et les renvoie sous la forme d'un dictionnaire.

`reduce_node` permet de traiter chaque nœud et sa liste de voisins dans le graphe pour émettre des relations mises à jour.
1. Initialisation :
	- `node, neighbors = line` permet de récupérer le nœud et ses voisins ;
	- La valeur minimale parmi les voisins (`min_node`) est initialisée avec la clef actuelle (`key`).

2. Trouver la valeur minimale :
	- Pour chaque voisin, on compare la valeur minimale actuelle avec la valeur du voisin ;
	- Si la valeur du voisin est inférieure à la valeur minimale actuelle, la valeur minimale est mise à jour.

3. Émettre les relations mises à jour :
	- Si la valeur minimale trouvée (`min_node`) est strictement inférieure à la clef (`key`), cela signifie qu'une mise à jour est nécessaire ;
	- Pour chaque voisin, une relation entre le nœud et la valeur minimale est émise.

**Remarque importante** : Au cours de l'algorithme `reduce_node` des relations temporaires sont émises. Ces relations sont ensuite fusionnées pour obtenir les relations finales. En effet, si un nœud `A` est connecté à un nœud `B` et que `B` est connecté à un nœud `C`, alors `A` est connecté à `C`. C'est pourquoi les relations temporaires sont émises avant d'être fusionnées.
Ces relations supplémentaires permettent d'abord de d'accélerer la convergence. C'est-à-dire qu'elles permettent de propager rapidement l'information du nœud minimal à travers le graphe. Ensuite, l'ajout temporaire de relations évite de devoir parcourir le graphe complet pour rechercher les connexions indirectes. Et enfin, cela permet de s'assurer que tous les nœuds connaissent leur composant minimal, c'est-à-dire que chaque nœud connaît le plus petit nœud de son composant connecté. Cette connection se limite à deux nœuds. Si un nœud `A` est connecté à un nœud `B` et que `B` est connecté à un nœud `C` et que `C` est connecté à un nœud `D`, alors `A` n'est pas connecté à `D`. En effet, l'algorithme ne permet pas de connecter des nœuds à plus de deux nœuds.


In [54]:
def reduce_node(node_neighbors):
    key, neighbors = node_neighbors
    min_node = key
    value_list = list(neighbors)

    # Trouver la valeur minimale
    for neighbor in neighbors:
        if neighbor < min_node:
            min_node = neighbor

    emissions = []
    if min_node < key:
        # Émettre la relation (key, min_node)
        emissions.append((key, min_node))
        # Émettre pour les autres valeurs
        for neighbor in value_list:
            if neighbor != min_node:
                emissions.append((neighbor, min_node))
    return emissions

In [ ]:
node_neighbors = ("B", ["A", "C", "D"])

[('B', 'A'), ('C', 'A'), ('D', 'A')]


1. Étape 1 : Trouver `min_node` = "A" ;
2. Étape 2 : Émettre :
	- ("B", "A")  (relation entre B et A)
	- ("C", "A")  (relation entre C et A)
	- ("D", "A")  (relation entre D et A)

In [56]:
relations = reduce_node(node_neighbors)
print(relations)

[('B', 'A'), ('C', 'A'), ('D', 'A')]


In [ ]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType
from pyspark.sql.functions import col, when, least, array, collect_set, explode

class GraphSpark:
    def __init__(self, input_data=None, sc=None):
        """
        Initialise le graphe avec un DataFrame ou un dictionnaire.
        """
        # Initialisation de SparkSession
        self.spark = SparkSession.builder.getOrCreate()
        self.sc = sc or self.spark.sparkContext

        # Définir un schéma vide pour comparaison
        empty_schema = StructType([
            StructField("src", StringType(), True),
            StructField("dst", StringType(), True),
        ])

        # Vérifier si l'entrée est un dictionnaire
        if isinstance(input_data, dict):
            edges = [
                (src, dst)
                for src, neighbors in input_data.items()
                for dst in neighbors
            ]
            self.graph_df = self.spark.createDataFrame(edges, ["src", "dst"])

        # Vérifier si l'entrée est un DataFrame PySpark
        elif isinstance(input_data, self.spark.createDataFrame([], empty_schema).__class__):
            self.graph_df = input_data

        # Lever une erreur si l'entrée est invalide
        else:
            raise ValueError("L'entrée doit être un dictionnaire ou un DataFrame PySpark")

    def __str__(self):
        """
        Représentation textuelle du graphe.
        """
        # Trier le DataFrame par la colonne 'src'
        sorted_edges = self.graph_df.orderBy("src", "dst").collect()
        s = ""
        for edge in sorted_edges:
            s += f"{edge['src']} -> {edge['dst']}\n"
        return s

    def __eq__(self, other):
        """
        Vérifie l'égalité entre deux graphes en comparant leurs DataFrames.
        """
        if not isinstance(other, GraphSpark):
            return False

        # Comparer les DataFrames par leurs données collectées et triées
        self_data = self.graph_df.sort("src", "dst").collect()
        other_data = other.graph_df.sort("src", "dst").collect()

        return self_data == other_data

    def mapper(self):
        """
        Étape Mapper : Crée un graphe bidirectionnel à partir du graphe actuel.
        """
        # Ajouter des relations dans les deux sens (bidirectionnel)
        bidirectional_graph = self.graph_df.unionByName(
            self.graph_df.select(col("dst").alias("src"), col("src").alias("dst"))
        ).distinct()

        # Retourner un nouveau graphe
        return GraphSpark(input_data=bidirectional_graph, sc=self.sc)

    def reduce(self, graph_df, debug=False):
        """
        Trouver le nœud minimal pour chaque clé et générer les relations mises à jour.
        """
        # Trouver le nœud minimal pour chaque source
        graph_with_min = graph_df.groupBy("src").agg(
            collect_set("dst").alias("neighbors")
        ).withColumn("min_node", least(col("src"), explode(col("neighbors"))))

        # Ajouter les nouvelles relations en propageant min_node
        updated_graph = graph_with_min.select(
            col("src").alias("src"),
            col("min_node").alias("dst")
        ).union(
            graph_with_min.select(
                explode(col("neighbors")).alias("src"),
                col("min_node").alias("dst")
            )
        ).distinct()

        if debug:
            print("Graph après réduction :")
            updated_graph.show()

        return updated_graph

    def reducer(self, graph_df, debug=False):
        """
        Appliquer la fonction reduce à tout le DataFrame pour obtenir le graphe mis à jour.
        """
        return self.reduce(graph_df, debug=debug)

# Initialisation de la session Spark
spark = SparkSession.builder.appName("GraphConversion").getOrCreate()

# Structure d'entrée
input_dict = {
    "A": ["B"],
    "B": ["C", "D"],
    "D": ["E"],
    "F": ["G"],
    "G": ["H"],
}

# Convertir le dictionnaire en une liste de tuples (src, dst)
edges = [(src, dst) for src, neighbors in input_dict.items() for dst in neighbors]

# Créer un DataFrame PySpark avec les colonnes "src" et "dst"
edges_df = spark.createDataFrame(edges, ["src", "dst"])

G = GraphSpark(edges_df)

print(G)

G_mapper = G.mapper()

print(G_mapper)

G_reducer = G.reduce(G_mapper.graph_df)

print(G_reducer)